
# Project Success Prediction Analytics

This project demonstrates a synthetic data analysis pipeline designed for aspiring Business Analysts, Program Managers, and Data Analysts. The goal is to perform exploratory data analysis and build a predictive model to estimate the likelihood of project success using a realistic synthetic dataset. The project is structured to show increasing levels of difficulty and showcases key skills such as data cleaning, visualization, feature engineering, and model evaluation.

## Dataset

The dataset (`synthetic_project_data.csv`) contains 1,000 synthetic project records with the following fields:

- **project_id**: Unique identifier for each project.
- **project_type**: Category of the project (e.g., Software, Infrastructure, Marketing).
- **region**: Geographic region where the project is executed.
- **industry**: Industry sector associated with the project.
- **budget**: Total budget allocated to the project (USD).
- **team_size**: Number of people in the project team.
- **duration_months**: Project duration in months.
- **stakeholder_engagement_score**: Score representing stakeholder engagement (0-100).
- **risk_score**: Risk assessment score (0-100).
- **complexity_score**: Complexity assessment score (0-100).
- **num_issues**: Number of issues encountered during the project.
- **prior_success_rate**: Historical success rate for similar projects (0-1).
- **success**: Target variable indicating whether the project was successful (1) or not (0).

## Objectives

1. Load and explore the synthetic dataset to understand its structure and properties.
2. Visualize the distribution of key variables and investigate relationships with project success.
3. Build a predictive model to estimate project success and evaluate its performance.
4. Provide recommendations on model improvement and further analysis.


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier

# Load dataset
projects_df = pd.read_csv('data/synthetic_project_data.csv')

# Display the first few rows
projects_df.head()


In [ ]:

# Summary statistics
projects_df.describe(include='all')


In [ ]:

# Visualize distribution of numeric features and target
numeric_cols = ['budget', 'team_size', 'duration_months', 'stakeholder_engagement_score',
                'risk_score', 'complexity_score', 'num_issues', 'prior_success_rate']
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for idx, col in enumerate(numeric_cols):
    ax = axes[idx//4, idx%4]
    sns.histplot(projects_df[col], kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

# Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(projects_df[numeric_cols + ['success']].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

# Boxplot of risk_score by success
plt.figure(figsize=(6, 4))
sns.boxplot(data=projects_df, x='success', y='risk_score')
plt.title('Risk Score by Project Success')
plt.xlabel('Success (1=Yes, 0=No)')
plt.ylabel('Risk Score')
plt.show()


In [ ]:

# Separate features and target
X = projects_df.drop(columns=['success', 'project_id'])
y = projects_df['success']

# Identify categorical and numerical columns
categorical_cols = ['project_type', 'region', 'industry']
numerical_cols = [col for col in X.columns if col not in categorical_cols]

# Preprocess data: one-hot encode categorical variables
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ], remainder='passthrough'
)

# Define model
model = RandomForestClassifier(n_estimators=200, random_state=42)

# Create pipeline
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Fit the model
clf.fit(X_train, y_train)

# Predict on test set
y_pred = clf.predict(X_test)

# Evaluate model
print('Accuracy:', accuracy_score(y_test, y_pred))
print('
Classification Report:
', classification_report(y_test, y_pred))
